# 09 — Spectrum Encoder & Differential k-mer Scoring

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | [`03` SVM with kernel](03_svc_with_kernel.ipynb) | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | [`06` Weighted (WGKM) kernel](06_weighted_kernel.ipynb) | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | `**09**` Spectrum encoder & NB | [`10` Gappy encoder](10_gappy_encoder.ipynb) | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | [`12` Shuffler & chunker](12_shuffler_and_chunker.ipynb)

This vignette demonstrates:
1. Using `SpectrumEncoder` to convert sequences into a sparse CSR matrix of k-mer counts.
2. Using `DifferentialKmerScorer` (Multinomial Naive Bayes) to identify enriched k-mers, with auto-generated negatives via `KmerShuffler`.

**See also:** [12_shuffler_and_chunker.ipynb](12_shuffler_and_chunker.ipynb) for background-model details.

In [1]:
import numpy as np
from kmer.encoders import SpectrumEncoder
from kmer.models import DifferentialKmerScorer
from kmer.perturb import KmerShuffler, Chunker

## 1. Encode sequences with SpectrumEncoder

In [2]:
seqs = ['ACGTACGTACGT', 'TTTTAAAACCCC', 'GCGCGCGCGCGC']
enc = SpectrumEncoder(k=4, canonical_rc=True)
X = enc.fit_transform(seqs)
print('Shape:', X.shape)
print('Format:', X.format)
print('Feature names[:5]:', enc.get_feature_names_out()[:5])
print('Total counts per seq:', X.sum(axis=1).A1)

Shape: (3, 136)
Format: csr
Feature names[:5]: ['AAAA' 'AAAC' 'AAAG' 'AAAT' 'AACA']
Total counts per seq: [9 9 9]


## 2. Differential k-mer scoring

In [3]:
def random_seq(n):
    return "".join(np.random.choice(list("ACGT"), size=n))

np.random.seed(0)

positives = [random_seq(50) + 'ATTCCG' + random_seq(50) for i in range(200)]

scorer = DifferentialKmerScorer(
    featurizer=SpectrumEncoder(k=4),
    background=KmerShuffler(k=1, seed=42),
)
scorer.fit(positives)

print('Top 5 enriched k-mers:')
print(sorted(scorer.kmer_scores_.items(), key=lambda x: -x[1])[:5])

Top 5 enriched k-mers:
[('TCCG', 1.2259517110447113), ('ATTC', 1.1716374236830003), ('TTCC', 1.046787220803524), ('GATT', 0.7363193524251539), ('CCGT', 0.6241543090729937)]


In [4]:
test_seqs = ['TTCCGATTCCGATTCC', 'TTTTAAAAGGGGCCCC', 'GCGCGCGCGCGCGCGC']
scores = scorer.decision_function(test_seqs)
for s, sc in zip(test_seqs, scores):
    print(f'  {s}: {sc:.3f}')

  TTCCGATTCCGATTCC: 10.742
  TTTTAAAAGGGGCCCC: -1.518
  GCGCGCGCGCGCGCGC: -0.482


## 3. Effect of background stringency

In [5]:
for bg_k in range(1, 4+1):
    s = DifferentialKmerScorer(
        featurizer=SpectrumEncoder(k=6, canonical_rc=True),
        background=KmerShuffler(k=bg_k, seed=42),
    )
    s.fit(positives)
    top = sorted(s.kmer_scores_.items(), key=lambda x: -x[1])[:5]
    print(f'background k={bg_k}: top 5 = {dict(top)}')

background k=1: top 5 = {'ATTCCG': 2.7175289450567544, 'CCCTAA': 2.0149030205422642, 'TCCGAA': 1.992430164690207, 'GATTCC': 1.9771626925594177, 'CCGTAC': 1.8971199848858813}
background k=2: top 5 = {'ATTCCG': 2.791636917210476, 'CATTCC': 2.093234863812172, 'CAAATG': 2.0149030205422642, 'CCGGAA': 1.9299098077088725, 'TGTACA': 1.7917594692280545}
background k=3: top 5 = {'ACTCCT': 2.1400661634962708, 'CGCGCG': 2.0794415416798353, 'ATTCCG': 1.8302257500558516, 'TGGCCA': 1.7047480922384253, 'AACCCA': 1.6094379124340996}
background k=4: top 5 = {'CCTTGC': 1.7917594692280545, 'GCTCCC': 1.7917594692280545, 'ACGCGT': 1.3862943611198908, 'AAGGAC': 1.38629436111989, 'AACACT': 1.2237754316221165}


In [6]:
for bg_k in range(1, 4+1):
    s = DifferentialKmerScorer(
        featurizer=SpectrumEncoder(k=6, canonical_rc=True),
        background=Chunker(min_size=bg_k, max_size=bg_k+2, flip_strand=True, flip_strand_prob=0.5),
    )
    s.fit(positives)
    top = sorted(s.kmer_scores_.items(), key=lambda x: -x[1])[:5]
    print(f'chunking k∈[{bg_k}, {bg_k+2}]: top 5 = {dict(top)}')

chunking k∈[1, 3]: top 5 = {'ATTCCG': 2.7175289450567544, 'GATTCC': 2.0949457282158015, 'GCGGAA': 1.8281271133989296, 'AATTCC': 1.7707060600302231, 'CCGGAA': 1.729239112246721}
chunking k∈[2, 4]: top 5 = {'ATTCCG': 2.6485360735698027, 'ACGGAA': 1.7764919970972661, 'GATTCC': 1.6094379124341005, 'CTCACA': 1.6094379124340996, 'CCGGAA': 1.5621850275835554}
chunking k∈[3, 5]: top 5 = {'ACATAC': 1.8971199848858813, 'CTCTTA': 1.871802176901591, 'ACGAGA': 1.7917594692280554, 'ATTCCG': 1.6430142079677044, 'CGAAAG': 1.6094379124341005}
chunking k∈[4, 6]: top 5 = {'ATTCCG': 1.4647659765613863, 'GTTAAC': 1.38629436111989, 'CATTGG': 1.2992829841302616, 'AAATCT': 1.203972804325936, 'ACCTCT': 1.1786549963416455}
